# CLOY Processing

This notebook is for you to work with the CLOY data folder.

In [2]:
%pip install pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 29.9 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 38.8 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install textblob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.0/625.0 kB 9.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.2/807.2 kB 14.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [textblob]7/8 [textblob]le]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import json
import math
import pandas as pd
from textblob import TextBlob

In [10]:
threads = []
with open("data/RedditExplorer/cloy_chronological.jsonl", encoding="utf-8") as f:
    for line in f:
        threads.append(json.loads(line))

print("Number of threads:", len(threads))

Number of threads: 3075


In [9]:
import os

for folder, subfolders, files in os.walk("."):
    if "cloy_chronological.jsonl" in files:
        print(os.path.join(folder, "cloy_chronological.jsonl"))

./data/RedditExplorer/cloy_chronological.jsonl


In [11]:
for t in threads:
    if t["post_id"] == "1d6vjw4":
        thread = t

print(thread.keys())
print("Title:", thread["title"])
print("Number of top-level comments:", len(thread["comments"]))
comment = thread["comments"][0]
print(comment.keys())
print("Comment:", comment["comment_body"])
reply = comment["replies"][0]
print(reply.keys())
print("Reply:", reply["comment_body"])
reply_to_reply = reply["replies"][0]
print("Reply to the reply:", reply_to_reply["comment_body"])

dict_keys(['post_id', 'timestamp', 'author', 'subreddit', 'title', 'selftext', 'url', 'score', 'permalink', 'comments'])
Title: If CLOY is #1, what's your #2?
Number of top-level comments: 66
dict_keys(['comment_id', 'parent_id', 'author', 'comment_body', 'score', 'created_utc', 'replies'])
Comment: King the land
dict_keys(['comment_id', 'parent_id', 'author', 'comment_body', 'score', 'created_utc', 'replies'])
Reply: About to finish it, my wife says that’s her best
Reply to the reply: So glad you guys liked it ❤️😍


In [12]:
SKIP = ["",
    "[deleted]",
    "[removed]",
    "This placeholder thread was generated to preserve historical comments whose parent post data is missing.",]

In [13]:
def get_comment_text(comments):
    text = ""
    for comment in comments:
        if comment["comment_body"] not in SKIP:
            text = text + " " + comment["comment_body"]
        replies = comment.get("replies", [])
        text = text + " " + get_comment_text(replies)
    return text
def get_thread_text(thread):
    text = ""
    if not thread["title"].startswith("DUMMY Thread"):
        text = thread["title"]
    if thread["selftext"] not in SKIP:
        text = text + " " + thread["selftext"]
    text = text + " " + get_comment_text(thread["comments"])
    return text
print(get_thread_text(thread)[:500])

If CLOY is #1, what's your #2? Clearly we all have excellent taste, so I'm wondering how that extends past CLOY?

I've seen some people list a bunch of different shows they like as well, and I've taken note of a few, but I'm really interested in seeing if people can just name one other show that really stands out for them like CLOY does.

For me the only thing that's come close so far is **Extraordinary Attorney Woo** - very different of course, but just struck the right notes with me.

So! What


## Make vectors

In [ ]:
def text_to_vector(text, stop_words=[]):
    words= TextBlob(text.lower()).words
    vector={}
    for word in words:
        if word.isalpha() and word not in stop_words:
            vector[word]=vector.get(word, 0) + 1
    return vector
vectors={}
titles={}
for thread in threads:
    thread_id = thread["post_id"]
    vectors[thread_id] = text_to_vector(get_thread_text(thread))
    titles[thread_id] = thread["title"]
print(len(vectors), "vectors")
print(vectors["1d6vjw4"])

MissingCorpusError: 
Looks like you are missing some required data for this feature.

To download the necessary data, simply run

    python -m textblob.download_corpora

or use the NLTK downloader to download the missing data: http://nltk.org/data.html
If this doesn't fix the problem, file an issue at https://github.com/sloria/TextBlob/issues.


In [16]:
def cosine_sim(v1, v2):
    dot_product = 0
    for word in v1:
        if word in v2:
            dot_product = dot_product + v1[word] * v2[word]

    length1 = math.sqrt(sum(count ** 2 for count in v1.values()))
    length2 = math.sqrt(sum(count ** 2 for count in v2.values()))

    if length1 == 0 or length2 == 0:
        return 0
    return dot_product / (length1 * length2)

In [17]:
def build_similarity_table(vectors):
    ids = list(vectors.keys())
    similarity = {}
    for thread_id in ids:
        similarity[thread_id] = {}
    for i in range(len(ids)):
        for j in range(i, len(ids)):
            score = cosine_sim(vectors[ids[i]], vectors[ids[j]])
            similarity[ids[i]][ids[j]] = score
            similarity[ids[j]][ids[i]] = score
        if i % 500 == 0:
            print(i, "threads done")

    return pd.DataFrame(similarity)